In [1]:
# ==========================================
# PHASE 6 - NEXT PURCHASE PREDICTION
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ------------------------------------------
# Load Data
# ------------------------------------------

df = pd.read_csv("../data/cleaned_online_retail.csv")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Dataset Shape:", df.shape)

# ------------------------------------------
# Define Snapshot Date
# ------------------------------------------

snapshot_date = df["InvoiceDate"].max() - pd.Timedelta(days=90)

# ------------------------------------------
# Split Data
# ------------------------------------------

past = df[df["InvoiceDate"] < snapshot_date]
future = df[df["InvoiceDate"] >= snapshot_date]

# ------------------------------------------
# Feature Engineering (Past Data)
# ------------------------------------------

features = past.groupby("CustomerID").agg({

    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "InvoiceNo": "nunique",
    "Revenue": "sum",
    "Quantity": "sum"

}).reset_index()

features.columns = [
    "CustomerID",
    "Recency",
    "Frequency",
    "Monetary",
    "TotalQuantity"
]

# ------------------------------------------
# Target Creation
# ------------------------------------------

future_customers = future["CustomerID"].unique()

features["WillPurchase"] = features["CustomerID"].apply(
    lambda x: 1 if x in future_customers else 0
)

print("\nTarget Distribution:")
print(features["WillPurchase"].value_counts())

# ------------------------------------------
# Features & Target
# ------------------------------------------

X = features[[
    "Recency",
    "Frequency",
    "Monetary",
    "TotalQuantity"
]]

y = features["WillPurchase"]

# ------------------------------------------
# Train Test Split
# ------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ------------------------------------------
# Models
# ------------------------------------------

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        random_state=42
    )
}

# ------------------------------------------
# Train & Evaluate
# ------------------------------------------

for name, model in models.items():

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print("\n", name)
    print("Accuracy:", accuracy_score(y_test, preds))
    print(classification_report(y_test, preds))

Dataset Shape: (392692, 9)

Target Distribution:
WillPurchase
1    1921
0    1449
Name: count, dtype: int64

 Random Forest
Accuracy: 0.6513353115727003
              precision    recall  f1-score   support

           0       0.58      0.64      0.61       284
           1       0.72      0.66      0.69       390

    accuracy                           0.65       674
   macro avg       0.65      0.65      0.65       674
weighted avg       0.66      0.65      0.65       674


 XGBoost
Accuracy: 0.6394658753709199
              precision    recall  f1-score   support

           0       0.57      0.59      0.58       284
           1       0.69      0.67      0.68       390

    accuracy                           0.64       674
   macro avg       0.63      0.63      0.63       674
weighted avg       0.64      0.64      0.64       674



In [2]:
from sklearn.ensemble import RandomForestClassifier
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    model,
    "../models/purchase_model.pkl"
)

print("Purchase model saved successfully!")

Purchase model saved successfully!


In [3]:
import joblib

clv_model = joblib.load("../models/clv_model.pkl")
purchase_model = joblib.load("../models/purchase_model.pkl")

print("CLV Model Features:")
print(clv_model.feature_names_in_)

print("\nPurchase Model Features:")
print(purchase_model.feature_names_in_)

CLV Model Features:
['Recency' 'Frequency' 'PastRevenue' 'AvgOrderValue' 'TotalQuantity'
 'AvgQuantity' 'ActiveDays' 'Segment_Champions' 'Segment_Lost Customers'
 'Segment_Loyal Customers' 'Segment_Potential Loyalists']

Purchase Model Features:
['Recency' 'Frequency' 'Monetary' 'TotalQuantity']
